In [1]:
# imports
import os
import requests
from bs4 import BeautifulSoup
from typing import List, Generator
from openai import OpenAI
import google.generativeai
from anthropic import Anthropic
import gradio as gr

In [39]:
# Config

# LLM models
GPT_MODEL = "gpt-4o-mini"
CLAUDE_MODEL = "claude-3-haiku-20240307"
GOOGLE_MODEL = "gemini-2.0-flash"
GOOGLE_API = "https://generativelanguage.googleapis.com/v1beta/openai/"

# LLM clients
openai = OpenAI()
claude = Anthropic()
gemini_openai = OpenAI(base_url=GOOGLE_API, api_key=os.getenv("GOOGLE_API_KEY"))


# LLM paremeters
MAX_TOKENS = 1000
TEMPERATURE = 0.7

# LLM instructions
system_message = "You are a helpful assistant"
user_message = "What is today's date?"

In [ ]:
def message_gpt(user_message: str) -> str:
    """
    Function to send a message to the GPT model and return the response.
    """
    response = openai.chat.completions.create(
        model=GPT_MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message},
        ],
    )
    return response.choices[0].message.content


message_gpt(
    user_message=user_message,
)

In [ ]:
def shout(text):
    print(f"Shout has been called with input: {text}")
    return text.upper()

shout("Hello, world!")

In [ ]:
view = gr.Interface(fn=shout, inputs="textbox", outputs="textbox", allow_flagging="never")
view.launch(share=True)

In [ ]:
view = gr.Interface(
    fn=message_gpt,
    inputs=[gr.Textbox(label="Your message:", lines=6)],
    outputs=[gr.Textbox(label="Response:", lines=8)],
    allow_flagging="never"
)
view.launch()

In [ ]:
system_message = "You are a helpful assistant that responds in markdown."

view = gr.Interface(
    fn=message_gpt,
    inputs=[gr.Textbox(label="Your message:")],
    outputs=[gr.Markdown(label="Response:")],
    flagging_mode="never",
)
view.launch()

In [10]:
def stream_gpt(user_message: str) -> Generator[str, None, None]:
    """
    Function to stream a message to the GPT model and return the response.
    """
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]
    stream = openai.chat.completions.create(
        model=GPT_MODEL,
        messages=messages,
        stream=True,
    )
    result = ""
    for chunk in stream:
        if chunk.choices[0].delta.content:
            result += chunk.choices[0].delta.content
            yield result

In [44]:
def stream_gemini(user_message: str) -> Generator[str, None, None]:
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}
    ]
    stream = gemini_openai.chat.completions.create(
        model=GOOGLE_MODEL,
        messages=messages,
        stream=True
    )

    result = ""
    for chunk in stream:
        if chunk.choices[0].delta.content:
            result += chunk.choices[0].delta.content
            yield result

In [11]:
view = gr.Interface(
    fn=stream_gpt,
    inputs=[gr.Textbox(label="Your message:")],
    outputs=[gr.Markdown(label="Response:")],
    flagging_mode="never",
)
view.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [16]:
def stream_claude(user_message: str) -> Generator[str, None, None]:
    """
    Function to stream a message to the Claude model and return the response.
    """
    response = claude.messages.stream(
        model=CLAUDE_MODEL,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
        system=system_message,
        messages=[
            {"role": "user", "content": user_message},
        ],
    )

    result = ""
    with response as stream:
        for text in stream.text_stream:
            result += text or ""
            yield result

In [52]:
view = gr.Interface(
    fn=stream_claude,
    inputs=[gr.Textbox(label="Your message:")],
    outputs=[gr.Markdown(label="Response:")],
    flagging_mode="never",
)
view.launch(share=True)

* Running on local URL:  http://127.0.0.1:7875
* Running on public URL: https://3169deb3480ae28054.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [37]:
def stream_model(user_message: str, model: str) -> Generator[str, None, None]:
    if model == "GPT":
        result = stream_gpt(user_message)
    elif model == "Claude":
        result = stream_claude(user_message)
    elif model == "Gemini":
        result = stream_gemini(user_message)
    else:
        raise ValueError("Unsupported model. Use 'GPT' or 'Claude'.")
    yield from result

In [45]:
view = gr.Interface(
    fn=stream_model,
    inputs=[
        gr.Textbox(label="Your message:"),
        gr.Dropdown(["GPT", "Claude", "Gemini"], label="Select model", value="GPT"),
    ],
    outputs=[gr.Markdown(label="Response:")],
    flagging_mode="never",
)
view.launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


In [31]:
class Website:
    url: str
    title: str
    text: str

    def __init__(self, url):
        self.url = url
        response = requests.get(url)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)
    
    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [21]:
system_mesage = "You are an assistant that analyzes the contents of a company website landing page \
and creates a short brochure about the company for prospective customers, investors and recruits. Respond in markdown."

In [49]:
def stream_brochure(company_name: str, url: str, model: str) -> Generator[str, None, None]:
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += Website(url).get_contents()
    if model == "GPT":
        result = stream_gpt(prompt)
    elif model == "Claude":
        result = stream_claude(prompt)
    elif model == "Gemini":
        result = stream_gemini(prompt)
    else:
        raise ValueError("Unknown model")
    yield from result


In [53]:
view = gr.Interface(
    fn=stream_brochure,
    inputs=[
        gr.Textbox(label="Company name:"),
        gr.Textbox(label="Landing page URL including http:// or https://"),
        gr.Dropdown(["GPT", "Claude", "Gemini"], label="Select model", value="GPT")],
    outputs=[gr.Markdown(label="Brochure")],
    flagging_mode="never"
)
view.launch(share=True)

* Running on local URL:  http://127.0.0.1:7876
* Running on public URL: https://d41137b8cb614f78f2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
